In [4]:
#!/usr/bin/env python3
"""
Oncoplots of the ER gene-set panels: primary vs second block (metastasis/recurrence), 12 patients.

Input : PCGR MAF per block  (<PCGR_DIR>/<block>_<SUFFIX>/<block>_<SUFFIX>.pcgr.grch38.maf)
        PCGR xlsx per block (<PCGR_DIR>/<block>_<SUFFIX>/<block>_<SUFFIX>.pcgr.grch38.xlsx), sheet "TMB"
        ER_scoring_gene_sets.xlsx, sheet "Genes (long)"  (columns: Gene set, Category, Gene)
For each gene set (panel) one oncoplot is drawn:
   columns = 12 patients as Primary (left half) and Second block (right half), same patient order
   rows    = panel genes with >= 1 non-silent mutation, ordered from high to low frequency
   top bar = TMB = PCGR TMB_ESTIMATE of the chosen TMB_MEASURE (default: coding + silent, mutations per Mb)
   right bars = % of patients mutated in Primary and in Second block (percentage written at the end of each bar)
   bottom row = Primary / Metastasis / Recurrence
Outputs (OUTDIR): oncoplot_<panel>.png/.pdf, panel_mutations.csv, panel_gene_summary.csv, panel_sample_counts.csv
"""
import os, re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Patch

# ----------------------------- PATHS (edit here) -----------------------------
PCGR_DIR     = "/ix1/alee/LO_LAB/Personal/Rahul/Okwin/WES/PCGR"
SUFFIX       = "S000021"
GENESET_XLSX = "/ix1/alee/LO_LAB/Personal/Rahul/Okwin/ER_scoring_gene_sets.xlsx"
OUTDIR       = "/ix1/alee/LO_LAB/Personal/Rahul/Okwin/WES/oncoplot_panels"
# -----------------------------------------------------------------------------
# TMB taken from the "TMB" sheet of each PCGR xlsx. Options: TMB_coding_and_silent | TMB_coding_non_silent | TMB_missense_only
TMB_MEASURE = "TMB_coding_and_silent"
TMB_LABEL   = {"TMB_coding_and_silent": "coding + silent", "TMB_coding_non_silent": "coding non-silent",
               "TMB_missense_only": "missense only"}[TMB_MEASURE]
EXOME_MB = 34            # exome size, used only for the raw non-silent count TMB written to panel_sample_counts.csv (for comparison)
MAX_GENES = 40            # genes shown per oncoplot (all mutated genes stay in the CSV tables)
EXTRA_PANEL = True        # also draw a panel of common breast cancer genes (ESR1, PIK3CA, TP53, ...)
EXTRA_GENES = ["ESR1", "PIK3CA", "TP53", "GATA3", "FOXA1", "CDH1", "ARID1A", "KMT2C", "MAP3K1",
               "PTEN", "AKT1", "ERBB2", "NF1", "RB1", "CCND1", "NCOR1"]

# 12 patients with exome on both sides: second block type (primary = <id>a, second = <id>b)
PATIENTS = {"PI_R_048": "Metastasis", "PI_R_052": "Recurrence", "PI_R_075": "Recurrence", "PI_R_090": "Metastasis",
            "PI_R_102": "Recurrence", "PI_R_103": "Recurrence", "PI_R_104": "Metastasis", "PI_R_105": "Metastasis",
            "PI_R_106": "Recurrence", "PI_R_111": "Metastasis", "PI_R_113": "Recurrence", "PI_R_117": "Recurrence"}
# order columns: metastasis first, then recurrence, then patient ID
ORDER = sorted(PATIENTS, key=lambda p: (PATIENTS[p] != "Metastasis", p))

NONSILENT = {"Missense_Mutation": "Missense", "Nonsense_Mutation": "Nonsense", "Frame_Shift_Del": "Frameshift",
             "Frame_Shift_Ins": "Frameshift", "In_Frame_Del": "In-frame indel", "In_Frame_Ins": "In-frame indel",
             "Splice_Site": "Splice site", "Translation_Start_Site": "Start lost", "Nonstop_Mutation": "Stop lost"}
COLORS = {"Missense": "#2E7D32", "Nonsense": "#C62828", "Frameshift": "#1565C0", "In-frame indel": "#8D6E63",
          "Splice site": "#F9A825", "Start lost": "#6A1B9A", "Stop lost": "#00838F", "Multi-hit": "#111111"}
GRP_COL = {"Primary": "#5B3A8C", "Second block": "#1F9E8F"}
SAMPLE_COL = {"Primary": "#5B3A8C", "Metastasis": "#E8A33D", "Recurrence": "#E0457B"}   # colour of each column
os.makedirs(OUTDIR, exist_ok=True)

# ---------------------- 1. read the 24 MAF files ------------------------------
rows = []
for pid in ORDER:
    for role, letter in (("Primary", "a"), ("Second block", "b")):
        block = f"{pid}{letter}"; sid = f"{block}_{SUFFIX}"
        f = os.path.join(PCGR_DIR, sid, f"{sid}.pcgr.grch38.maf")
        if not os.path.isfile(f):
            print("MISSING MAF:", f); continue
        d = pd.read_csv(f, sep="\t", comment="#", low_memory=False)
        d = d[d["Variant_Classification"].isin(NONSILENT)].copy()
        prot = "HGVSp_Short" if "HGVSp_Short" in d.columns else None
        out = pd.DataFrame({"patient": pid, "role": role, "block": block, "gene": d["Hugo_Symbol"],
                            "class": d["Variant_Classification"].map(NONSILENT),
                            "protein_change": d[prot].fillna("") if prot else ""})
        if {"t_alt_count", "t_depth"} <= set(d.columns):
            out["vaf"] = (pd.to_numeric(d["t_alt_count"], errors="coerce") / pd.to_numeric(d["t_depth"], errors="coerce")).round(3)
        rows.append(out)
muts = pd.concat(rows, ignore_index=True)
tmb_count = muts.groupby(["patient", "role"]).size().div(EXOME_MB)    # raw non-silent count / 34 Mb (comparison only, not plotted)
print(f"Non-silent mutations read: {len(muts)} from {muts['block'].nunique()} blocks")

# ---------------------- 1b. TMB from the PCGR xlsx ("TMB" sheet) ---------------
tmb, tmb_meta = {}, {}                                                # key = (patient, role)
for pid in ORDER:
    for role, letter in (("Primary", "a"), ("Second block", "b")):
        sid = f"{pid}{letter}_{SUFFIX}"
        fx = os.path.join(PCGR_DIR, sid, f"{sid}.pcgr.grch38.xlsx")
        tmb[(pid, role)] = np.nan; tmb_meta[(pid, role)] = {}
        if not os.path.isfile(fx):
            print("MISSING XLSX (TMB set to NaN, bar drawn as 0):", fx); continue
        t = pd.read_excel(fx, sheet_name="TMB")
        r = t[t["TMB_MEASURE"] == TMB_MEASURE]
        if r.empty:
            print(f"{TMB_MEASURE} not found in TMB sheet of {sid}; found: {t['TMB_MEASURE'].tolist()}"); continue
        r = r.iloc[0]
        tmb[(pid, role)] = float(r["TMB_ESTIMATE"])
        tmb_meta[(pid, role)] = dict(tmb_n_variants=r["TMB_N_VARIANTS"], tmb_dp_min=r["TMB_DP_MIN"], tmb_af_min=r["TMB_AF_MIN"],
                                     tmb_target_mb=r["TMB_TARGET_SIZE_MB"])
print(f"TMB ({TMB_MEASURE}) read for {sum(not np.isnan(v) for v in tmb.values())} of {len(tmb)} blocks")

# same gene + protein change in both blocks of a patient = shared variant
key = muts["gene"] + "|" + muts["protein_change"]
other = {"Primary": "Second block", "Second block": "Primary"}
sets = {(r.patient, r.role): set() for r in muts.itertuples()}
for r, k in zip(muts.itertuples(), key): sets[(r.patient, r.role)].add(k)
muts["shared_with_pair"] = [k in sets.get((r.patient, other[r.role]), set()) and r.protein_change != "" for r, k in zip(muts.itertuples(), key)]

# ---------------------- 2. gene panels ----------------------------------------
gl = pd.read_excel(GENESET_XLSX, sheet_name="Genes (long)")
panels = {name: set(g["Gene"].dropna().astype(str)) for name, g in gl.groupby("Gene set", sort=False)}
if EXTRA_PANEL: panels["Selected_breast_cancer_genes"] = set(EXTRA_GENES)

columns = [(p, "Primary") for p in ORDER] + [(p, "Second block") for p in ORDER]
xpos = {c: (i if c[1] == "Primary" else i + 1) for i, c in enumerate(columns)}      # one empty column between the two halves
def sample_type(c): return "Primary" if c[1] == "Primary" else PATIENTS[c[0]]

def call(classes):
    classes = list(classes)
    return classes[0] if len(set(classes)) == 1 and len(classes) == 1 else "Multi-hit"

all_mut, gene_rows, count_rows = [], [], []
for panel, genes in panels.items():
    m = muts[muts["gene"].isin(genes)].copy(); m.insert(0, "panel", panel); all_mut.append(m)
    cell = {(g, c): call(df["class"]) for (g, pid, role), df in m.groupby(["gene", "patient", "role"]) for c in [(pid, role)]}
    # gene table: patients mutated in primary / second / both (counts and % of the 12 patients)
    for g, d in m.groupby("gene"):
        pp = set(d[d.role == "Primary"].patient); ps = set(d[d.role == "Second block"].patient)
        gene_rows.append(dict(panel=panel, gene=g, n_primary=len(pp), n_second=len(ps), n_both=len(pp & ps),
                              pct_primary=round(100 * len(pp) / len(ORDER), 1), pct_second=round(100 * len(ps) / len(ORDER), 1),
                              n_patients_any=len(pp | ps), shared_protein_change=int(d.shared_with_pair.any())))
    for c in columns:
        count_rows.append(dict(panel=panel, patient=c[0], role=c[1], sample_type=sample_type(c),
                               tmb_measure=TMB_MEASURE, tmb_pcgr_per_mb=round(float(tmb.get(c, np.nan)), 3),
                               tmb_nonsilent_count_per_mb=round(float(tmb_count.get(c, 0)), 3), **tmb_meta.get(c, {}),
                               panel_genes=len(genes),
                               genes_mutated=int(m[(m.patient == c[0]) & (m.role == c[1])]["gene"].nunique())))
    # ---------------- 3. draw the oncoplot ----------------
    freq = m.drop_duplicates(["gene", "patient", "role"]).groupby("gene").size().reset_index(name="n")   # empty panel -> empty table
    genes_show = freq.sort_values(["n", "gene"], ascending=[False, True])["gene"].tolist()[:MAX_GENES]
    ng = len(genes_show); xmax = max(xpos.values()) + 0.6; nP = len(ORDER)
    FS = 14
    # panel heights in inches, so small panels (few genes) keep the same spacing as large ones
    H_TITLE, H_TOP, H_MAIN, H_ANN, H_LEG = 0.95, 1.4, 0.30 * max(ng, 3), 0.95, 1.25
    FIG_H = H_TITLE + H_TOP + H_MAIN + H_ANN + H_LEG + 0.5
    fig = plt.figure(figsize=(13, FIG_H))
    gs = fig.add_gridspec(4, 2, height_ratios=[H_TOP, H_MAIN, H_ANN, H_LEG], width_ratios=[1, 0.3],   # wider right panel so the % labels fit
                          hspace=0.12, wspace=0.04, left=0.10, right=0.985, top=1 - H_TITLE / FIG_H, bottom=0.01)
    axT, ax, axA, axL, axR = (fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[2, 0]),
                              fig.add_subplot(gs[3, :]), fig.add_subplot(gs[1, 1]))
    axL.axis("off")
    if ng == 0:
        for a_ in (ax, axT, axA, axR): a_.axis("off")
        fig.text(0.5, 0.5, f"{panel}: no non-silent mutations in the {len(genes)} panel genes", ha="center", fontsize=15)
    else:
        for yi, g in enumerate(genes_show):
            for c in columns:
                v = cell.get((g, c))
                ax.add_patch(Rectangle((xpos[c] - 0.45, yi - 0.42), 0.9, 0.84, fc=COLORS[v] if v else "#EEEAF2", ec="none"))
        ax.set_xlim(-0.6, xmax); ax.set_ylim(ng - 0.5, -0.5)
        ax.set_yticks(range(ng)); ax.set_yticklabels(genes_show, fontsize=FS, style="italic"); ax.set_xticks([])
        for s_ in ax.spines.values(): s_.set_visible(False)
        ax.tick_params(length=0)
        # top annotation: PCGR TMB of each sample (chosen TMB_MEASURE), coloured by sample type; missing value -> 0
        axT.bar([xpos[c] for c in columns], [0 if np.isnan(tmb.get(c, np.nan)) else tmb[c] for c in columns],
                color=[SAMPLE_COL[sample_type(c)] for c in columns], width=0.85)
        axT.set_xlim(-0.6, xmax); axT.set_xticks([]); axT.set_ylabel(f"TMB (mut/Mb)\n{TMB_LABEL}", fontsize=FS - 2)
        axT.tick_params(axis="y", labelsize=FS - 1)
        for s_ in ("top", "right"): axT.spines[s_].set_visible(False)
        for lab, xc in (("Primary", (nP - 1) / 2), ("Second block", nP + 1 + (nP - 1) / 2)):
            axT.text(xc, 1.03, lab, transform=axT.get_xaxis_transform(), ha="center", va="bottom", fontsize=FS + 1, fontweight="bold", color=GRP_COL[lab])
        # right bars: % of patients mutated in each group (denominator = 12 patients per group), % written at the end of each bar
        gr = {r["gene"]: r for r in gene_rows if r["panel"] == panel}
        for yi, g in enumerate(genes_show):
            for ypos, k_, col in ((yi - 0.18, "n_primary", GRP_COL["Primary"]),
                                  (yi + 0.18, "n_second",  GRP_COL["Second block"])):
                pct = 100 * gr[g][k_] / nP                       # count -> percentage
                axR.barh(ypos, pct, height=0.34, color=col)
                if pct > 0:                                      # no label for empty bars
                    axR.text(pct + 2, ypos, f"{pct:.0f}%", va="center", ha="left", fontsize=FS - 5)
        axR.set_ylim(ng - 0.5, -0.5); axR.set_xlim(0, 115); axR.set_yticks([])
        axR.set_xticks([0, 50, 100])                             # labels sit in the free space after 100
        axR.set_xlabel("Patients mutated (%)", fontsize=FS); axR.tick_params(axis="x", labelsize=FS - 1)
        for s_ in ("top", "right", "left"): axR.spines[s_].set_visible(False)
        axR.legend(handles=[Patch(fc=GRP_COL["Primary"], label="Primary"), Patch(fc=GRP_COL["Second block"], label="Second block")],
                   loc="lower left", bbox_to_anchor=(0.0, 1.02), frameon=False, fontsize=FS - 1, handlelength=1.0)
        # (legend sits above the bar panel, in the empty space next to the TMB bar, so it cannot cover any bars)
        # bottom annotation: Primary / Metastasis / Recurrence for every column; patient label under each column
        for c in columns:
            axA.add_patch(Rectangle((xpos[c] - 0.45, 0.05), 0.9, 0.9, fc=SAMPLE_COL[sample_type(c)], ec="none"))
            axA.text(xpos[c], -0.12, c[0].replace("PI_R_", ""), ha="center", va="top", fontsize=FS - 1, rotation=90)
        axA.set_xlim(-0.6, xmax); axA.set_ylim(-2.2, 1); axA.axis("off")
        # two separate legends: mutation type (left) and sample type (right)
        used = sorted({v for v in cell.values()}, key=list(COLORS).index)
        l1 = axL.legend(handles=[Patch(fc=COLORS[u], label=u) for u in used], title="Mutation type", title_fontsize=FS,
                        loc="upper left", bbox_to_anchor=(0.02, 1.0), ncol=4, frameon=False, fontsize=FS, handlelength=1.2, columnspacing=1.4)
        axL.add_artist(l1)
        l2 = axL.legend(handles=[Patch(fc=SAMPLE_COL[t], label=t) for t in SAMPLE_COL], title="Sample type", title_fontsize=FS,
                        loc="upper left", bbox_to_anchor=(0.66, 1.0), ncol=1, frameon=False, fontsize=FS, handlelength=1.2)
        for lg in (l1, l2):                                   # bold, left-aligned titles (works in old and new matplotlib)
            lg.get_title().set_fontweight("bold")
            try: lg._legend_box.align = "left"
            except Exception: pass
        fig.suptitle(f"{panel}  ({len(genes)} genes; {len(freq)} with a non-silent mutation" + (f", top {MAX_GENES} shown)" if len(freq) > MAX_GENES else ")"),
                     x=0.5, y=1 - 0.08 / FIG_H, fontsize=FS + 2, fontweight="bold")
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(OUTDIR, f"oncoplot_{re.sub(r'[^A-Za-z0-9_]+', '_', panel)}.{ext}"), dpi=300, bbox_inches="tight", pad_inches=0.1)
    plt.close(fig)
    print(f"{panel}: {len(genes)} genes, {len(freq)} mutated, {int(freq['n'].sum()) if len(freq) else 0} gene-sample hits")

pd.concat(all_mut).to_csv(os.path.join(OUTDIR, "panel_mutations.csv"), index=False)
pd.DataFrame(gene_rows).sort_values(["panel", "n_patients_any"], ascending=[True, False]).to_csv(os.path.join(OUTDIR, "panel_gene_summary.csv"), index=False)
pd.DataFrame(count_rows).to_csv(os.path.join(OUTDIR, "panel_sample_counts.csv"), index=False)
print("Outputs in", OUTDIR)

Non-silent mutations read: 30949 from 24 blocks
TMB (TMB_coding_and_silent) read for 24 of 24 blocks
er_upreg: 59 genes, 27 mutated, 59 gene-sample hits
er_pos_corr: 103 genes, 61 mutated, 149 gene-sample hits
er_neg_corr: 59 genes, 38 mutated, 80 gene-sample hits
estrogene_v2_esr1_mutant_up: 66 genes, 36 mutated, 76 gene-sample hits
estrogene_v2_esr1_mutant_down: 63 genes, 39 mutated, 88 gene-sample hits
Y537S_unique_genes: 420 genes, 190 mutated, 397 gene-sample hits
D538G_unique_genes: 364 genes, 198 mutated, 527 gene-sample hits
low_dose_tamoxifen_repressed: 193 genes, 138 mutated, 304 gene-sample hits
high_dose_tamoxifen_repressed: 169 genes, 99 mutated, 197 gene-sample hits
TamR_up: 107 genes, 73 mutated, 175 gene-sample hits
TamR_down: 102 genes, 70 mutated, 171 gene-sample hits
Selected_breast_cancer_genes: 16 genes, 16 mutated, 69 gene-sample hits
Outputs in /ix1/alee/LO_LAB/Personal/Rahul/Okwin/WES/oncoplot_panels
